## Tl;dr Short version

For an ordinary transition, the essential workflow is:

```python
from pathlib import Path
from pexl import schema

result = schema.create(
    Path("../data/exports/schema_XXX.xlsx"),
    dataset_dir=Path("../data/schemas/schema_XXX"),
    version="XXX",
    replace_files=True
    )
```

Then:

1. review the diff against the previous schema,
2. inspect added variables,
3. point `pexl.schema.current` to the generated module,
4. load a representative project,
5. inspect the new variables,
6. run the tests.

In detail:


# Transition PEExcel to a new schema version

This notebook documents the developer workflow for moving `pexl` from one PEExcel schema version to the next.

It assumes that the new schema workbook has already been exported from PEExcel using the **``Export_Schema()``** macro in VBA Module `a_Export`. The workflow then:

1. creates the versioned schema dataset and generated Python bindings,
2. compares the new schema with the previous version,
3. inspects newly added variables,
4. activates the new generated schema,
5. checks those new variables in a real PEExcel project,
6. runs the test suite.

The intended audience is colleagues working with the PEExcel Python codebase.

## 1. Configure the version transition

Keep the version names explicit. The schema workbook is the source exported from PEExcel; the schema directories are the versioned CSV datasets used for comparison and code generation.

In [1]:
from pathlib import Path
import pandas as pd

from pexl import schema

OLD_VERSION = "v1_12_7"
NEW_VERSION = "1.13.1_dev"

SCHEMA_XLSX = Path(f"../data/exports/schema_{NEW_VERSION}.xlsx")
OLD_SCHEMA_DIR = Path(f"../data/schemas/{OLD_VERSION}")
NEW_SCHEMA_DIR = Path(f"../data/schemas/{NEW_VERSION}")

SCHEMA_XLSX, OLD_SCHEMA_DIR, NEW_SCHEMA_DIR

(WindowsPath('../data/exports/schema_1.13.1_dev.xlsx'),
 WindowsPath('../data/schemas/v1_12_7'),
 WindowsPath('../data/schemas/1.13.1_dev'))

Before continuing, check that:

- the PEExcel schema/input-structure version was incremented,
- the schema workbook was freshly exported,
- `IN`, `OUT`, `CHART_metadata`, and `CHART_types` are present in the export,
- `OLD_SCHEMA_DIR` points to the version currently used as the comparison baseline.

## 2. Create the new schema bindings directly from the Excel export

`schema.create(...)` performs the complete conversion:

```text
schema_<version>.xlsx
        ↓
versioned schema CSV dataset
        ↓
generated variable/schema module
        +
generated chart/report module
```

The returned object contains the paths of the generated artifacts.

`overwrite_dataset=True` replaces files in `NEW_SCHEMA_DIR`, so verify the paths above before running the cell.

In [2]:
result = schema.create(
    SCHEMA_XLSX,
    dataset_dir=NEW_SCHEMA_DIR,
    version=NEW_VERSION,
    replace_files=True,
)

result

SchemaBuildResult(source_path=WindowsPath('../data/exports/schema_1.13.1_dev.xlsx'), dataset_dir=WindowsPath('../data/schemas/1.13.1_dev'), generated_dir=WindowsPath('C:/Users/schneids/code/peexcel/src/pexl/schema/generated'), schema_module=WindowsPath('C:/Users/schneids/code/peexcel/src/pexl/schema/generated/excel_1_13_1_dev.py'), report_module=WindowsPath('C:/Users/schneids/code/peexcel/src/pexl/schema/generated/reports_1_13_1_dev.py'), version='1_13_1_dev')

In [3]:
print("Source workbook :", result.source_path)
print("Schema dataset  :", result.dataset_dir)
print("Schema module   :", result.schema_module)
print("Report module   :", result.report_module)
print("Version         :", result.version)

Source workbook : ..\data\exports\schema_1.13.1_dev.xlsx
Schema dataset  : ..\data\schemas\1.13.1_dev
Schema module   : C:\Users\schneids\code\peexcel\src\pexl\schema\generated\excel_1_13_1_dev.py
Report module   : C:\Users\schneids\code\peexcel\src\pexl\schema\generated\reports_1_13_1_dev.py
Version         : 1_13_1_dev


## 3. Compare the old and new schema

The existing schema diff gives the complete structural comparison of the two versioned datasets.

In [4]:
from pexl.validate.schema import diff_schema_dirs, schema_diff_to_markdown
from IPython.display import display, Markdown

diff_result = diff_schema_dirs(
    old_dir=OLD_SCHEMA_DIR,
    new_dir=NEW_SCHEMA_DIR,
)

display(
    Markdown(
        schema_diff_to_markdown(
            diff_result,
            value_format="None",
        )
    )
)

# Schema Diff

- **Old:** `..\data\schemas\v1_12_7`
- **New:** `..\data\schemas\1.13.1_dev`
- **Key:** `None`

## Dataset files

- Old file count: 3
- New file count: 5
- Added: ['CHART_metadata', 'CHART_types']
- Removed: []

### Warnings
- ⚠ Tables added to schema: ['CHART_metadata', 'CHART_types']

## IN

### Columns
- Added: []
- Removed: []
- Common count: 13

### Rows
Old count: 691
New count: 692
- **New Error count: 6**
- Added/~~Removed~~ var_name:
   - PV_roof_area_percentage



### Potential Errors (#)

| var_name | column | error |
|---|---|---|
| fGHG_grid_column | Formel | VERGLEICH(fGHG_grid_profile;fGHG[#Kopfzeilen];0) |
| fPE_grid_column | Formel | VERGLEICH(fPE_grid_profile;fPE[#Kopfzeilen];0) |
| flex_Signals_selected_column | Formel | VERGLEICH(FLEX_signal_name;Signals[#Kopfzeilen];0) |
| rcp1_dh | Formel | WENNFEHLER(WENN(FINDEN("rneuerbar";SVERWEIS(heat_th2_carrier_type;cf_constant;7;FALSCH));rcp1_renewable;#NV);rcp1_fossil) |
| rcp2_dh | Formel | WENNFEHLER(WENN(FINDEN("rneuerbar";SVERWEIS(heat_th2_carrier_type;cf_constant;7;FALSCH));rcp2_renewable;#NV);rcp2_fossil) |
| rcp3_dh | Formel | WENNFEHLER(WENN(FINDEN("rneuerbar";SVERWEIS(heat_th2_carrier_type;cf_constant;7;FALSCH));rcp3_renewable;#NV);rcp3_fossil) |





## OUT

### Columns
- Added: []
- Removed: []
- Common count: 13

### Rows
Old count: 452
New count: 506
- **New Error count: 0**
- Added/~~Removed~~ var_name:
   - EUIdhw_1el
  - EUIdhw_2el
  - EUIdhw_th_total
  - Ec_1el
  - Ec_3el
  - Eh_1el
  - Eh_3el
  - PEI_cf_density_neg
  - PEI_cf_density_pos
  - PV_peak_grid_feedin
  - PV_peak_grid_feedin_date
  - Qc_min_0fc
  - Qced_1el
  - Qced_2th
  - Qced_3el
  - Qdhw_1_distr_losses
  - Qdhw_1_drawoff
  - Qdhw_1_generation_losses
  - Qdhw_1_storage_losses
  - Qdhw_1_tap
  - Qdhw_1_total
  - Qdhw_2_distr_losses
  - Qdhw_2_drawoff
  - Qdhw_2_generation_losses
  - Qdhw_2_storage_losses
  - Qdhw_2_tap
  - Qdhw_2_total
  - Qdhw_distr_losses
  - Qdhw_drawoff
  - Qdhw_generation_losses
  - Qdhw_storage_charge
  - Qdhw_storage_losses
  - Qdhw_tap
  - Qenv_c
  - Qenv_c_1el
  - Qenv_c_3el
  - Qenv_dhw
  - Qenv_dhw_1
  - Qenv_dhw_2
  - Qenv_h
  - Qenv_h_1el
  - Qenv_h_3el
  - Qh_distr_losses
  - Qh_min_wasteheat
  - Qheb_1el
  - Qheb_2th
  - Qheb_3el
  - Qheb_4th
  - Qloss_c_generation
  - Qloss_c_th2_generation
  - Qloss_h_generation
  - Qloss_h_th2_generation
  - Qloss_h_th4_generation
  - test_EUI_balance
  - test_GWP_sum
  - test_heat_balance_seasonal_mismatch
  - test_legacy_GWP_sum
  - ~~Edhw_1_th~~
  - ~~Edhw_2_th~~
  - ~~Edhw_th~~
  - ~~PEI_cf_density~~

- ⚠ Duplicates (new): ['Qh_distr_losses']

### Potential Errors (#)
- None





## SIM

### Columns
- Added: []
- Removed: []
- Common count: 5

### Rows
Old count: 394
New count: 401
- **New Error count: 3**
- Added/~~Removed~~ var_name:
   - DHW_1_tap_kW
  - DHW_2_tap_kW
  - Ec_1el
  - Ec_3el
  - Edhw_1el
  - Edhw_2el
  - Eh_1el
  - Eh_3el
  - Qenv_c_1el
  - Qenv_c_3el
  - Qenv_dhw_1
  - Qenv_dhw_2
  - Qenv_h_1el
  - Qenv_h_3el
  - Qh_distr_losses
  - Qhed_1el
  - Qhed_2th
  - Qhed_3el
  - Qhed_4th
  - Qhed_total
  - Spalte14
  - Test
  - ~~Batt_max_energy_output~~
  - ~~QH_distr_losses~~
  - ~~Qh_u2~~
  - ~~Qheb_1el~~
  - ~~Qheb_2th~~
  - ~~Qheb_3el~~
  - ~~Qheb_4th~~
  - ~~Qheb_total~~
  - ~~Spalte192~~
  - ~~Spalte80~~
  - ~~Spalte81~~
  - ~~Spalte82~~
  - ~~Spalte83~~
  - ~~Spalte84~~
  - ~~Spalte85~~


### Potential Errors (#)

| var_name | column | error |
|---|---|---|
| Signal | Formula | =INDEX(Signals,ROW()-ROW(Signals[#Headers]),flex_Signals_selected_column) |
| Ti0cooled | Formula | =IF(NFA_cooled/NFA_total>0,Tsetheat_min,#N/A) |
| Ti0uncooled | Formula | =IF(1-NFA_cooled/NFA_total>0,Tsetheat_min,#N/A) |





## CHART_metadata

- Status: **added**
- Rows: {'count': 73}
- Columns: ['chart_name', 'label_de', 'var_name', 'role', 'order', 'color', 'pattern']

## CHART_types

- Status: **added**
- Rows: {'count': 7}
- Columns: ['chart_name', 'tab_name', 'title', 'chart_type']

## Audit summary

- Raw audit present (old): True
- Raw audit present (new): True

### Empty var_name exclusions (converter)

**Old:**
```json
{'IN': {'empty_var_name_count': 94}, 'OUT': {'empty_var_name_count': 110}, 'SIM': {'empty_var_name_count': 0}}
```
**New:**
```json
{'IN': {'empty_var_name_count': 94}, 'OUT': {'empty_var_name_count': 106}, 'SIM': {'empty_var_name_count': None}, 'CHART_metadata': {'empty_var_name_count': None}, 'CHART_types': {'empty_var_name_count': None}}
```

Review the diff before activating the new schema. In particular, check for:

- unintentionally removed variables,
- renamed `var_name` values,
- changed units or metadata,
- variables that moved between `IN` and `OUT`,
- unexpected changes in the chart metadata exports.

## 4. List variables added since the previous schema

For a quick review, compare the canonical `var_name` values in `IN.csv` and `OUT.csv`.

This is deliberately independent of the internal structure of `diff_result`, so it is also useful in ad-hoc migration checks.

In [8]:
def read_schema_variables(schema_dir: Path) -> pd.DataFrame:
    frames = []

    for source in ("IN", "OUT"):
        path = schema_dir / f"{source}.csv"
        df = pd.read_csv(path, sep=";", encoding="utf-8-sig")
        df = df.copy()
        df["source"] = source
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


old_vars = read_schema_variables(OLD_SCHEMA_DIR)
new_vars = read_schema_variables(NEW_SCHEMA_DIR)

added_var_names = sorted(
    set(new_vars["var_name"].dropna())
    - set(old_vars["var_name"].dropna())
)

print(f"Added variables: {len(added_var_names)}")
added_var_names

Added variables: 58


['EUIdhw_1el',
 'EUIdhw_2el',
 'EUIdhw_th_total',
 'Ec_1el',
 'Ec_3el',
 'Eh_1el',
 'Eh_3el',
 'PEI_cf_density_neg',
 'PEI_cf_density_pos',
 'PV_peak_grid_feedin',
 'PV_peak_grid_feedin_date',
 'PV_roof_area_percentage',
 'Qc_min_0fc',
 'Qced_1el',
 'Qced_2th',
 'Qced_3el',
 'Qdhw_1_distr_losses',
 'Qdhw_1_drawoff',
 'Qdhw_1_generation_losses',
 'Qdhw_1_storage_losses',
 'Qdhw_1_tap',
 'Qdhw_1_total',
 'Qdhw_2_distr_losses',
 'Qdhw_2_drawoff',
 'Qdhw_2_generation_losses',
 'Qdhw_2_storage_losses',
 'Qdhw_2_tap',
 'Qdhw_2_total',
 'Qdhw_distr_losses',
 'Qdhw_drawoff',
 'Qdhw_generation_losses',
 'Qdhw_storage_charge',
 'Qdhw_storage_losses',
 'Qdhw_tap',
 'Qenv_c',
 'Qenv_c_1el',
 'Qenv_c_3el',
 'Qenv_dhw',
 'Qenv_dhw_1',
 'Qenv_dhw_2',
 'Qenv_h',
 'Qenv_h_1el',
 'Qenv_h_3el',
 'Qh_distr_losses',
 'Qh_min_wasteheat',
 'Qheb_1el',
 'Qheb_2th',
 'Qheb_3el',
 'Qheb_4th',
 'Qloss_c_generation',
 'Qloss_c_th2_generation',
 'Qloss_h_generation',
 'Qloss_h_th2_generation',
 'Qloss_h_th4_genera

Show the new variables together with the metadata that is most useful during review:

In [6]:
review_columns = [
    col
    for col in [
        "source",
        "var_name",
        "label_de",
        "unit",
        "domain",
        "measure",
        "entity_group",
        "entity_key",
        "ka",
        "comment",
    ]
    if col in new_vars.columns
]

added_variables = (
    new_vars.loc[new_vars["var_name"].isin(added_var_names), review_columns]
    .sort_values(["source", "var_name"])
    .reset_index(drop=True)
)

added_variables

,source,var_name,label_de,unit,domain,measure,entity_group,entity_key,ka,comment
0,IN,PV_roof_area_percentage,% Dachfläche für PV genützt,ratio,PV,PV_roof_area_percentage,NaN,NaN,2.0,NaN
1,OUT,EUIdhw_1el,Strom → WW-System 1,kWh Strom/m²NGFa,dhw,NaN,NaN,NaN,NaN,NaN
2,OUT,EUIdhw_2el,Strom → WW-System 2,kWh Strom/m²NGFa,dhw,NaN,NaN,NaN,NaN,NaN
3,OUT,EUIdhw_th_total,Thermische EE,kWh Wärme/m²NGFa,dhw,NaN,NaN,NaN,NaN,NaN
4,OUT,Ec_1el,Strom → Kühlsystem 1 (elektrisch),kWh Strom/m²NGFa,cooling,NaN,cooling,system_1,NaN,NaN
5,OUT,Ec_3el,Strom → Kühlsystem 3 (elektrisch),kWh Strom/m²NGFa,cooling,NaN,cooling,system_3,NaN,NaN
6,OUT,Eh_1el,Strom → Heizsystem 1 (elektrisch),kWh Strom/m²NGFa,heating,NaN,heating,system_1,NaN,NaN
7,OUT,Eh_3el,Strom → Heizsystem 3 (elektrisch),kWh Strom/m²NGFa,heating,NaN,heating,system_3,NaN,NaN
8,OUT,PEI_cf_density_neg,Kontext bauliche Dichte negativ,kWhPEges./m²NGFa,primary_energy_balance,supply,context_factor,density,3.0,NaN
9,OUT,PEI_cf_density_pos,Kontext bauliche Dichte positiv,kWhPEges./m²NGFa,primary_energy_balance,supply,context_factor,density,3.0,NaN


## 5. Smoke-test the generated schema module before activating it

The generated file can be imported directly from its returned path. This verifies that code generation produced valid Python without changing the active `pexl.schema.current` binding yet.

In [9]:
import importlib.util
import sys

module_name = "pexl_schema_candidate"

spec = importlib.util.spec_from_file_location(
    module_name,
    result.schema_module,
)

candidate_schema = importlib.util.module_from_spec(spec)

# Required before exec_module(), notably for @dataclass
sys.modules[module_name] = candidate_schema

spec.loader.exec_module(candidate_schema)

print("Generated schema version:", candidate_schema.SCHEMA_VERSION)
print("Variables:", len(candidate_schema.ATTR_NAME_MAP))

Generated schema version: 1_13_1_dev
Variables: 1197


In [10]:
# Confirm that all newly added var_names were generated.
missing_from_generated = [
    var_name
    for var_name in added_var_names
    if var_name not in candidate_schema.ATTR_NAME_MAP
]

missing_from_generated

[]

The expected result is an empty list.

At this stage the generated schema exists, but normal `Project` imports still use the module referenced by `pexl.schema.current`.

## 6. Activate the new schema

Update `src/pexl/schema/current.py` so that it imports the newly generated schema module.

For example, if the generated file is:

```text
excel_1_13_1_dev.py
```

the active binding should point to that module.

After changing `current.py`, restart the Python kernel or reload the affected `pexl` modules before testing a project. This avoids mixing classes and metadata from two schema versions in one interpreter session.

## 7. Load a project created with the new PEExcel version

Use a representative PEExcel project/export that was saved or exported with the new version.

Change `PROJECT_PATH` to a suitable test project.

In [13]:
from pexl import Project

PROJECT_PATH = Path("../data/exports/ka_project_backup_v1_13.xlsx")

project = Project.from_excel(PROJECT_PATH)

project

<Project scenarios=53 warnings=24 source='..\\data\\exports\\ka_project_backup_v1_13.xlsx'>

In [14]:
print("Scenarios:", len(project))
print("Warnings :", len(project.warnings))

project.warnings[:10]

Scenarios: 53
Warnings : 24


["Column 'Default': conventional project name 'Default' does not match project_name='Neues Projekt'",
 "Column 'Default': conventional scenario name 'Default' does not match project_scenario_name='Basis'",
 "Column 'EFHo | nach 2020': conventional project name 'EFHo' does not match project_name='EFHk'",
 "Column 'EFHo | nach 2020': conventional scenario name 'nach 2020' does not match project_scenario_name='BT2 DE Südtiroler FLEX1,7xPV'",
 "Column 'DLG1k | Büro - 2010 - 2030': conventional scenario name 'Büro - 2010 - 2030' does not match project_scenario_name='Büro - bis 1978'",
 "Column 'DLG1g | Büro - 2010 - 2030': conventional scenario name 'Büro - 2010 - 2030' does not match project_scenario_name='Büro - bis 1978'",
 "Column 'DLG2k | Büro - 2010 - 2030': conventional scenario name 'Büro - 2010 - 2030' does not match project_scenario_name='Büro - bis 1978'",
 "Column 'DLG2g | Büro - 2010 - 2030': conventional scenario name 'Büro - 2010 - 2030' does not match project_scenario_name='

## 8. Inspect the newly added variables in the project

This is a small migration-specific introspection: for every variable that did not exist in the previous schema, show its metadata and its value in one representative scenario.

In [16]:
scenario = project[1]

rows = []

for var_name in added_var_names:
    attr_name = candidate_schema.ATTR_NAME_MAP.get(var_name)

    if attr_name is None:
        continue

    meta = getattr(scenario.meta, attr_name, None)

    rows.append(
        {
            "var_name": var_name,
            "attr_name": attr_name,
            "source": getattr(meta, "source", None),
            "label_de": getattr(meta, "label_de", None),
            "unit": getattr(meta, "unit", None),
            "value": getattr(scenario.v, attr_name, None),
        }
    )

added_in_project = pd.DataFrame(rows)

added_in_project

,var_name,attr_name,source,label_de,unit,value
0,EUIdhw_1el,EUIdhw_1el,OUT,Strom → WW-System 1,kWh Strom/m²NGFa,NaN
1,EUIdhw_2el,EUIdhw_2el,OUT,Strom → WW-System 2,kWh Strom/m²NGFa,NaN
2,EUIdhw_th_total,EUIdhw_th_total,OUT,Thermische EE,kWh Wärme/m²NGFa,NaN
3,Ec_1el,Ec_1el,OUT,Strom → Kühlsystem 1 (elektrisch),kWh Strom/m²NGFa,NaN
4,Ec_3el,Ec_3el,OUT,Strom → Kühlsystem 3 (elektrisch),kWh Strom/m²NGFa,NaN
5,Eh_1el,Eh_1el,OUT,Strom → Heizsystem 1 (elektrisch),kWh Strom/m²NGFa,NaN
6,Eh_3el,Eh_3el,OUT,Strom → Heizsystem 3 (elektrisch),kWh Strom/m²NGFa,NaN
7,PEI_cf_density_neg,PEI_cf_density_neg,OUT,Kontext bauliche Dichte negativ,kWhPEges./m²NGFa,0.000000e+00
8,PEI_cf_density_pos,PEI_cf_density_pos,OUT,Kontext bauliche Dichte positiv,kWhPEges./m²NGFa,3.996058e+01
9,PV_peak_grid_feedin,PV_peak_grid_feedin,OUT,PV Einspeisung Spitzenleistung,kW,0.000000e+00


If values need to be checked across all scenarios, create a compact scenario-by-variable table:

In [17]:
added_values = pd.DataFrame(
    {
        s.column_name: {
            var_name: getattr(
                s.v,
                candidate_schema.ATTR_NAME_MAP[var_name],
                None,
            )
            for var_name in added_var_names
            if var_name in candidate_schema.ATTR_NAME_MAP
        }
        for s in project
    }
).T

added_values.index.name = "column_name"

added_values

,EUIdhw_1el,EUIdhw_2el,EUIdhw_th_total,Ec_1el,Ec_3el,Eh_1el,Eh_3el,PEI_cf_density_neg,PEI_cf_density_pos,PV_peak_grid_feedin,...,Qheb_4th,Qloss_c_generation,Qloss_c_th2_generation,Qloss_h_generation,Qloss_h_th2_generation,Qloss_h_th4_generation,test_EUI_balance,test_GWP_sum,test_heat_balance_seasonal_mismatch,test_legacy_GWP_sum
column_name,,,,,,,,,,,,,,,,,,,,,
Default,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.256741,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.025047e-13,-117.904659,NaN,0.0
Forsthausgasse | BT2 KON FW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,39.960583,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.563194e-13,0.000000,1.477663e-01,0.0
Campo Breitenlee | Basis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.242371,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-2.309264e-13,NaN,NaN,0.0
EFHo | nach 2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-125.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,534.169868,NaN,0.0
EFHk | nach 2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-103.031470,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,0.0
MFHo | nach 2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-25.034128,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,0.0
MFHk | nach 2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-25.034128,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,0.0
GWBk | nach 2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.256741,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,0.0
GWBg | nach 2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.256741,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,0.0


This table is particularly useful for detecting cases where:

- a variable was generated correctly but not imported from the workbook,
- a new variable is unexpectedly empty in all scenarios,
- a variable is populated only for some variants,
- a renamed or moved variable no longer maps to the expected project data.

## 9. Optional: inspect new variables through the normal view API

Once the new schema is active, normal metadata-based selection should work for the new variables as well.

For individual variables, direct value access remains the clearest smoke test:

In [18]:
for var_name in added_var_names[:10]:
    attr_name = candidate_schema.ATTR_NAME_MAP[var_name]
    print(
        f"{var_name:40s}",
        getattr(scenario.v, attr_name, None),
    )

EUIdhw_1el                               nan
EUIdhw_2el                               nan
EUIdhw_th_total                          nan
Ec_1el                                   nan
Ec_3el                                   nan
Eh_1el                                   nan
Eh_3el                                   nan
PEI_cf_density_neg                       0
PEI_cf_density_pos                       39.960583066314804
PV_peak_grid_feedin                      0


## 10. Run the test suite

Run the schema and Excel I/O tests after activating the new version. From the repository root, for example:

```bash
pytest
```

At minimum, verify:

- generated schema imports,
- schema metadata and named-variable access,
- project import/export,
- schema diff functionality,
- report/chart binding generation,
- representative PEExcel project loading.

## 11. Migration checklist

A PEExcel schema transition is complete when:

- [ ] PEExcel input/schema version was incremented.
- [ ] A fresh schema workbook was exported from PEExcel.
- [ ] `schema.create(...)` created the new CSV dataset.
- [ ] Variable/schema Python bindings were generated.
- [ ] Chart/report bindings were generated.
- [ ] Old/new schema diff was reviewed.
- [ ] Added variables were reviewed explicitly.
- [ ] The generated schema module imports successfully.
- [ ] `pexl.schema.current` points to the new version.
- [ ] A representative project from the new PEExcel version loads successfully.
- [ ] Newly added variables contain the expected project values.
- [ ] Test suite passes.
- [ ] Generated schema/report files and schema dataset are committed with the version change.